In [95]:
%matplotlib inline
from __future__ import division
import matplotlib
import numpy as np
from pylab import *
from mpl_toolkits.mplot3d import Axes3D
import os
matplotlib.rcParams.update({"axes.formatter.limits": (-4,4)})
plotStyles={"markersize":12,"markeredgewidth":3.0,"linewidth":3.0}
stepStyles={"markersize":12,"markeredgewidth":3.0,"linewidth":3.0,"where":"post"}
np.seterr(divide='ignore',invalid='ignore')
pass

### Setup the tests.

In [106]:
import h5py
import math
testNames=["first_order_birth_death_mol_min",
           "first_order_birth_death_mol_sec",
           "first_order_birth_death_particle_min",
           "first_order_birth_death_particle_sec"]
test_names_bash_list=" ".join(testNames)

In [119]:
%%bash
rm -rf tmp && mkdir tmp

### Define the tests.

In [120]:
def test_first_order_birth_death(testOutputFilename):
    fp = h5py.File(testOutputFilename, "r")
    nr=fp["/Model/Reaction/"].attrs["numberReactions"]
    ns=fp["/Model/Reaction/"].attrs["numberSpecies"]
    D=fp["/Model/Reaction/DependencyMatrix"]
    C=fp["/Model/Reaction/InitialSpeciesCounts"]
    K=fp["/Model/Reaction/ReactionRateConstants"]
    R=fp["/Model/Reaction/ReactionTypes"]
    S=fp["/Model/Reaction/StoichiometricMatrix"]
    if nr != 1: raise Exception("numberReactions: incorrect value")
    if ns != 1: raise Exception("numberSpecies: incorrect value")
    if D.shape != (1,1): raise Exception("D: incorrect shape")
    if D[0,0] != 1: raise Exception("D: incorrect value")
    if C.shape != (ns,): raise Exception("C: incorrect shape")
    if C[0] != 1000: raise Exception("C: incorrect value")
    if K.shape != (nr,10): raise Exception("K: incorrect shape")
    if K[0,0] != 10.0: raise Exception("K: incorrect value")
    for i in range(1,10):
        if not math.isnan(K[0,i]): raise Exception("K: incorrect nan value")
    if R.shape != (nr,): raise Exception("R: incorrect shape")
    if R[0] != 1: raise Exception("R: incorrect value")
    if S.shape != (1,1): raise Exception("S: incorrect shape")
    if S[0,0] != -1: raise Exception("S: incorrect value")
    fp.close()

### Execute the SBML imports.

In [121]:
%%bash -s "$test_names_bash_list"
for testName in $1; do
    inputFilename=${testName}.xml
    outputFilename=tmp/${testName}.lm
    rm -f ${outputFilename}* && lm_sbml_import --copasi ${outputFilename} ${inputFilename} 2>&1 > ${outputFilename}.log
done;
echo "Finished."

Finished.


### Run the tests.

In [122]:
testMethods = {"first_order_birth_death_mol_min": test_first_order_birth_death,
              "first_order_birth_death_mol_sec": test_first_order_birth_death,
              "first_order_birth_death_particle_min": test_first_order_birth_death,
              "first_order_birth_death_particle_sec": test_first_order_birth_death}

for testName in testNames:
    try:
        testOutputFilename="tmp/%s.lm"%(testName)
        #testMethods[testName](testOutputFilename)
        test_first_order_birth_death(testOutputFilename)
    except Exception as e:
        print "%-60s : FAILED with:"%("["+testName+"]"),e
    except:
        print "%-60s : FAILED with: Unknown exception"%("["+testName+"]")
    else:
        print "%-60s : passed."%("["+testName+"]")

[first_order_birth_death_mol_min]                            : passed.
[first_order_birth_death_mol_sec]                            : passed.
[first_order_birth_death_particle_min]                       : passed.
[first_order_birth_death_particle_sec]                       : passed.
